# Agent Actions Trace Notebook

목표:
- Agent가 어떤 툴을 어떤 순서로 썼는지 보기
- 중간 추론(reasoning_chain)과 최종 판단을 같이 보기
- (옵션) 실시간 실행에서 툴 입력/출력까지 캡처하기

In [ ]:
import os
import json
from typing import Any
from textwrap import shorten
from IPython.display import display, Markdown

try:
    import pandas as pd
except Exception:
    pd = None

from data.db import DB_PATH, get_conn

print(f"DB_PATH: {DB_PATH}")
print(f"DB exists: {os.path.exists(DB_PATH)}")

In [ ]:
def _parse_json_list(value: Any) -> list:
    if value is None:
        return []
    if isinstance(value, list):
        return value
    text = str(value).strip()
    if not text:
        return []
    try:
        obj = json.loads(text)
        if isinstance(obj, list):
            return obj
    except Exception:
        pass
    return [text]


def load_agent_signals(limit: int = 50, only_with_trace: bool = True):
    where = "WHERE tool_sequence IS NOT NULL AND tool_sequence != ''" if only_with_trace else ""
    query = f"""
        SELECT
            id,
            created_at,
            stock_code,
            stock_name,
            signal_type,
            triggered_conditions,
            tool_sequence,
            reasoning_chain,
            claude_opinion
        FROM signals
        {where}
        ORDER BY id DESC
        LIMIT ?
    """
    with get_conn() as conn:
        rows = [dict(r) for r in conn.execute(query, (limit,)).fetchall()]

    for r in rows:
        tools = _parse_json_list(r.get("tool_sequence"))
        thoughts = _parse_json_list(r.get("reasoning_chain"))
        r["tool_list"] = tools
        r["reasoning_list"] = thoughts
        r["tool_count"] = len(tools)
        r["reasoning_count"] = len(thoughts)
        r["flow_preview"] = " -> ".join(tools[:6])
        if len(tools) > 6:
            r["flow_preview"] += " -> ..."
    return rows

In [ ]:
rows = load_agent_signals(limit=30, only_with_trace=True)
print(f"rows: {len(rows)}")

preview_cols = [
    "id", "created_at", "stock_code", "stock_name", "signal_type",
    "tool_count", "reasoning_count", "flow_preview"
]
preview_rows = [{k: r.get(k) for k in preview_cols} for r in rows]

if pd is not None:
    display(pd.DataFrame(preview_rows))
else:
    for x in preview_rows:
        print(x)

In [ ]:
def show_signal_flow(signal_id: int, thought_preview: int = 260, opinion_preview: int = 700):
    with get_conn() as conn:
        row = conn.execute(
            """
            SELECT
                id, created_at, stock_code, stock_name, signal_type,
                triggered_conditions, tool_sequence, reasoning_chain, claude_opinion
            FROM signals
            WHERE id = ?
            """,
            (signal_id,),
        ).fetchone()

    if not row:
        print(f"signal_id={signal_id} not found")
        return

    r = dict(row)
    tools = _parse_json_list(r.get("tool_sequence"))
    thoughts = _parse_json_list(r.get("reasoning_chain"))
    cond = r.get("triggered_conditions") or ""

    header = [
        f"## Signal #{r['id']} - {r.get('stock_name')} ({r.get('stock_code')})",
        f"- created_at: {r.get('created_at')}",
        f"- signal_type: {r.get('signal_type')}",
        f"- triggered_conditions: {cond}",
        f"- tool_count: {len(tools)} | reasoning_count: {len(thoughts)}",
    ]
    display(Markdown("\n".join(header)))

    n = max(len(tools), len(thoughts))
    if n == 0:
        display(Markdown("저장된 tool_sequence/reasoning_chain 이 없습니다."))
    else:
        lines = ["### Step-by-step"]
        for i in range(n):
            t = tools[i] if i < len(tools) else "(no tool)"
            th = thoughts[i] if i < len(thoughts) else "(no reasoning text captured)"
            th_short = shorten(str(th).replace("\n", " "), width=thought_preview, placeholder=" ...")
            lines.append(f"{i+1}. tool: `{t}`")
            lines.append(f"   - thought: {th_short}")
            lines.append("   - tool_output: (DB에는 단계별 출력이 저장되지 않음)")
        display(Markdown("\n".join(lines)))

    opinion = r.get("claude_opinion") or ""
    opinion_short = shorten(opinion.replace("\n", " "), width=opinion_preview, placeholder=" ...")
    display(Markdown("### Final Opinion"))
    print(opinion_short if opinion_short else "(empty)")

In [ ]:
# 예시: 가장 최근 trace 1건 보기
if rows:
    show_signal_flow(rows[0]["id"])
else:
    print("trace가 저장된 signal이 없습니다.")

## Optional: 실시간 Tool 입/출력까지 캡처

아래 셀은 `JudgmentAgent` 실행 중 `BaseAgent._execute`를 임시 패치해서
각 tool call의 입력/출력을 캡처합니다.

주의:
- OpenAI 키/브로커 API 키/네트워크 상태에 따라 실패할 수 있음
- 툴 출력이 길 수 있어 일부만 preview로 표시

In [ ]:
from worker.monitor import Signal
from worker.agents.base_agent import BaseAgent
from worker.agents.judgment_agent import JudgmentAgent


def build_signal_from_db(signal_id: int | None = None) -> Signal:
    with get_conn() as conn:
        if signal_id is None:
            row = conn.execute(
                "SELECT * FROM signals ORDER BY id DESC LIMIT 1"
            ).fetchone()
        else:
            row = conn.execute(
                "SELECT * FROM signals WHERE id = ?", (signal_id,)
            ).fetchone()

    if not row:
        raise ValueError("signals 테이블에 데이터가 없습니다.")

    r = dict(row)
    conds = [c.strip() for c in str(r.get("triggered_conditions") or "").split(",") if c.strip()]

    return Signal(
        stock_code=str(r.get("stock_code") or ""),
        stock_name=str(r.get("stock_name") or ""),
        current_price=int(r.get("current_price") or 0),
        triggered_conditions=conds,
        triggered_ids=[],
        rsi=r.get("rsi"),
        volume_ratio=r.get("volume_ratio"),
        in_portfolio=bool(r.get("in_portfolio")),
        signal_type=str(r.get("signal_type") or "both"),
    )


def run_live_trace(signal: Signal, max_steps: int = 6, max_tokens: int = 700):
    captured = []
    original_execute = BaseAgent._execute

    def traced_execute(self, name: str, inputs: dict):
        result = original_execute(self, name, inputs)
        captured.append({
            "tool": name,
            "inputs": inputs,
            "output": result,
        })
        return result

    BaseAgent._execute = traced_execute
    try:
        agent = JudgmentAgent(max_steps=max_steps, max_tokens=max_tokens)
        opinion = agent.run(signal)
        return {
            "opinion": opinion,
            "used_tools": agent.used_tools,
            "reasoning_chain": agent.reasoning_chain,
            "tool_calls": captured,
        }
    finally:
        BaseAgent._execute = original_execute


def show_live_trace(trace: dict, output_preview: int = 500):
    display(Markdown("### Live Trace Summary"))
    print("used_tools:", " -> ".join(trace.get("used_tools", [])) or "(none)")
    print("reasoning_steps:", len(trace.get("reasoning_chain", [])))
    print("tool_calls:", len(trace.get("tool_calls", [])))

    for idx, call in enumerate(trace.get("tool_calls", []), start=1):
        print(f"\n[{idx}] tool: {call['tool']}")
        print("inputs:", json.dumps(call.get("inputs", {}), ensure_ascii=False, default=str))
        out_text = json.dumps(call.get("output", {}), ensure_ascii=False, default=str)
        print("output:", shorten(out_text, width=output_preview, placeholder=" ..."))

    display(Markdown("### Final Opinion"))
    print(trace.get("opinion", ""))

In [ ]:
# 필요할 때만 실행
# sample_signal = build_signal_from_db()  # 또는 build_signal_from_db(signal_id=123)
# live = run_live_trace(sample_signal)
# show_live_trace(live)